In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score, classification_report

# ====== 데이터 로드 & one-hot ======
file_path = '/Users/leeseungbo/Desktop/학회 데이터(CSV)/5차 전처리 데이터(1).csv'
df = pd.read_csv(file_path)

target_names = ['사망발생', '중상발생', '경상발생']

X_raw = df.drop(columns=target_names)
X = pd.get_dummies(X_raw)

Y = df[target_names].astype(int)

# ====== (1) 멀티라벨 여부 체크: 한 행에서 1이 2개 이상이면 멀티라벨 ======
row_sum = Y.sum(axis=1)
n_multilabel_rows = int((row_sum >= 2).sum())
n_all = len(Y)

print("="*70)
print(f"멀티라벨(동시 1) 행 개수: {n_multilabel_rows} / {n_all}  ({n_multilabel_rows/n_all:.4%})")
print("행 합(row_sum) 분포:\n", row_sum.value_counts().sort_index())
print("="*70)

is_multilabel = n_multilabel_rows > 0

# ====== Train:Test = 7:3 ======
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42, stratify=Y  # 주의: multilabel에서는 stratify가 지원/의미가 제한적일 수 있음
)

# stratify 문제 발생 시(에러나거나 성능 왜곡 우려) 아래처럼 stratify=None로 바꾸세요.
# X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

# ====== (2) 멀티라벨이면 One-vs-Rest로 한번에 학습 ======
# 스크린샷 기준 하이퍼파라미터
base_estimator = xgb.XGBClassifier(
    max_depth=8,
    learning_rate=0.1,
    colsample_bytree=1.0,

    # n_estimators는 일단 고정(early stopping은 현재 7:3만 쓰면 val이 없어 불가)
    n_estimators=500,
    subsample=1.0,
    gamma=0.0,

    # 불균형은 label별로 다르지만 OneVsRest는 각 라벨별로 별도 모델이므로
    # scale_pos_weight를 고정하면 왜곡될 수 있음 → 여기서는 일단 미설정(=1.0)으로 둠
    # 필요하면 아래 “라벨별 spw 적용 버전”으로 바꾸는 게 더 정확함.
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

if is_multilabel:
    print("\n[모드] 멀티라벨(One-vs-Rest)로 학습합니다.\n")

    clf = OneVsRestClassifier(base_estimator, n_jobs=-1)
    clf.fit(X_train, Y_train)

    # 확률 예측: (n_samples, n_labels)
    Y_proba = clf.predict_proba(X_test)
    Y_pred = (Y_proba >= 0.5).astype(int)

    # 라벨별 ROC-AUC 출력 (각 라벨 독립 이진 AUC)
    print("라벨별 ROC-AUC")
    for j, name in enumerate(target_names):
        auc = roc_auc_score(Y_test.iloc[:, j], Y_proba[:, j])
        print(f"- {name}: {auc:.4f}")

    print("\n라벨별 classification_report (threshold=0.50)")
    for j, name in enumerate(target_names):
        print("\n" + "-"*60)
        print(f"[{name}]")
        print(classification_report(Y_test.iloc[:, j], Y_pred[:, j], digits=2))

else:
    print("\n[모드] 동시 1이 없으므로(비-멀티라벨), 기존처럼 라벨별 이진모델 3개가 적합합니다.\n")
    # 기존 방식(라벨별로 따로 학습)로 진행
    for name in target_names:
        y_train = Y_train[name]
        y_test = Y_test[name]

        model = base_estimator
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:, 1]
        pred = (proba >= 0.5).astype(int)

        auc = roc_auc_score(y_test, proba)
        print("\n" + "="*60)
        print(f"{name} ROC-AUC: {auc:.4f}")
        print(classification_report(y_test, pred, digits=2))


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

# ====== 데이터 로드 & one-hot ======
file_path = ㅊ
# 업로드 파일을 쓰려면 아래로 변경 가능:
# file_path = '/mnt/data/5차 전처리 데이터(1).csv'

df = pd.read_csv(file_path)

target_cols = ['사망발생', '중상발생', '경상발생']
X_raw = df.drop(columns=target_cols)
X = pd.get_dummies(X_raw)

# ====== 4클래스 y 생성 ======
# 0=비사상(3개 모두 0), 1=경상, 2=중상, 3=사망
y = np.zeros(len(df), dtype=int)
y[df['경상발생'].astype(int) == 1] = 1
y[df['중상발생'].astype(int) == 1] = 2
y[df['사망발생'].astype(int) == 1] = 3

class_names = {0: "비사상", 1: "경상", 2: "중상", 3: "사망"}

print("="*70)
print("4클래스 타깃 분포(y):")
print(pd.Series(y).value_counts().sort_index().rename(index=class_names))
print("="*70)

# ====== Train:Test = 7:3 ======
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test,  label=y_test)

# ====== 멀티클래스 XGBoost 학습 (native API) ======
params = {
    "objective": "multi:softprob",
    "num_class": 4,
    "eval_metric": "mlogloss",

    # (요청 반영) 스크린샷 기준
    "max_depth": 8,
    "eta": 0.1,
    "colsample_bytree": 1.0,

    # 기본값(원하면 추후 튜닝)
    "subsample": 1.0,
    "gamma": 0.0,

    "seed": 42
}

num_boost_round = 500
booster = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=num_boost_round,
    verbose_eval=False
)

# ====== 예측 ======
proba = booster.predict(dtest)      # (n_samples, 4)
y_pred = np.argmax(proba, axis=1)

# ====== (A) 4클래스 전체 평가 ======
print("\n" + "="*70)
print("[4클래스 전체] Accuracy:", accuracy_score(y_test, y_pred))
print("[4클래스 전체] classification_report")
print("="*70)
print(classification_report(
    y_test, y_pred, digits=3,
    target_names=[class_names[i] for i in range(4)]
))

print("\n[4클래스 전체] Confusion Matrix (rows=true, cols=pred)")
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])
cm_df = pd.DataFrame(
    cm,
    index=[f"T:{class_names[i]}" for i in range(4)],
    columns=[f"P:{class_names[i]}" for i in range(4)]
)
print(cm_df)

# ====== (B) 클래스별 one-vs-rest 평가 ======
# 각 클래스 확률을 따로 떼서 "해당 클래스 vs 나머지"로 평가
threshold = 0.5

print("\n" + "="*70)
print("[클래스별 one-vs-rest 평가] (threshold=0.50)")
print("="*70)

for c in range(4):
    y_true_bin = (y_test == c).astype(int)
    y_proba_bin = proba[:, c]

    # AUC (해당 클래스 vs 나머지)
    auc = roc_auc_score(y_true_bin, y_proba_bin)

    # threshold 기반 이진 예측
    y_pred_bin = (y_proba_bin >= threshold).astype(int)

    print("\n" + "-"*70)
    print(f"[{class_names[c]}] one-vs-rest | ROC-AUC={auc:.4f}")
    print(classification_report(y_true_bin, y_pred_bin, digits=3))


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ====== 데이터 로드 & one-hot ======
file_path = '/Users/leeseungbo/Desktop/학회 데이터(CSV)/5차 전처리 데이터(1).csv'
df = pd.read_csv(file_path)

target_cols = ['사망발생', '중상발생', '경상발생']
X_raw = df.drop(columns=target_cols)
X = pd.get_dummies(X_raw)

# ====== 4클래스 y 생성: 0=비사상, 1=경상, 2=중상, 3=사망 ======
y = np.zeros(len(df), dtype=int)
y[df['경상발생'].astype(int) == 1] = 1
y[df['중상발생'].astype(int) == 1] = 2
y[df['사망발생'].astype(int) == 1] = 3

class_names = {0:"비사상", 1:"경상", 2:"중상", 3:"사망"}

# ====== split: Train:Val:Test = 7:1:3 ======
def split_train_val_test(X, y, r_train=7, r_val=1, r_test=3, seed=42):
    total = r_train + r_val + r_test
    temp_size = (r_val + r_test) / total  # train vs (val+test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=temp_size, random_state=seed, stratify=y
    )
    val_frac_in_temp = r_val / (r_val + r_test)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(1 - val_frac_in_temp), random_state=seed, stratify=y_temp
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y, 7, 1, 3, seed=42)

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)
dtest  = xgb.DMatrix(X_test,  label=y_test)

# ====== 모델 학습(Precision 우선: 과적합 억제 쪽으로) ======
params = {
    "objective": "multi:softprob",
    "num_class": 4,
    "eval_metric": "mlogloss",

    # Precision 우선이면 깊게 가지 말고(오경보 감소), 샘플링으로 일반화
    "max_depth": 3,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "gamma": 0.05,
    "seed": 42
}

booster = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=3000,
    evals=[(dval, "val")],
    early_stopping_rounds=100,
    verbose_eval=False
)

# ====== Precision 우선 threshold 탐색 ======
# 목표 precision(원하면 조정)
P_TARGET_DEATH = 0.25    # 사망을 "사망"이라 할 때 precision 최소 목표
P_TARGET_SER   = 0.20    # 중상 precision 최소 목표

def precision_recall_for_class(y_true, proba_c, class_id, thr):
    # class_id를 양성으로 보는 one-vs-rest
    y_pos = (y_true == class_id).astype(int)
    y_hat = (proba_c >= thr).astype(int)

    tp = np.sum((y_hat == 1) & (y_pos == 1))
    fp = np.sum((y_hat == 1) & (y_pos == 0))
    fn = np.sum((y_hat == 0) & (y_pos == 1))

    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    return precision, recall

def find_threshold_by_precision(y_true, proba_c, class_id, p_target, thr_grid):
    # precision >= target을 만족하는 thr 중 recall이 최대인 thr 선택
    best = None
    for thr in thr_grid:
        p, r = precision_recall_for_class(y_true, proba_c, class_id, thr)
        if p >= p_target:
            if (best is None) or (r > best["recall"]):
                best = {"thr": thr, "precision": p, "recall": r}
    # 만족하는 게 없으면: precision 최대 thr로 대체
    if best is None:
        best_p = -1
        best_thr = 0.5
        best_r = 0.0
        for thr in thr_grid:
            p, r = precision_recall_for_class(y_true, proba_c, class_id, thr)
            if p > best_p:
                best_p, best_thr, best_r = p, thr, r
        return {"thr": best_thr, "precision": best_p, "recall": best_r, "met_target": False}
    best["met_target"] = True
    return best

# val 확률
val_proba = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))

thr_grid = np.linspace(0.05, 0.95, 91)

death_info = find_threshold_by_precision(y_val, val_proba[:, 3], 3, P_TARGET_DEATH, thr_grid)
ser_info   = find_threshold_by_precision(y_val, val_proba[:, 2], 2, P_TARGET_SER,   thr_grid)

t_death = death_info["thr"]
t_ser   = ser_info["thr"]

print("=== [VAL] threshold 선택 결과 (Precision 우선) ===")
print(f"사망: thr={t_death:.2f} | P={death_info['precision']:.3f}, R={death_info['recall']:.3f} | target_met={death_info['met_target']}")
print(f"중상: thr={t_ser:.2f}   | P={ser_info['precision']:.3f}, R={ser_info['recall']:.3f} | target_met={ser_info['met_target']}")

# ====== 결정 규칙(오경보 억제용) ======
def predict_with_priority(proba, t_death, t_ser):
    # proba: (n,4)
    pred = np.argmax(proba, axis=1)  # 기본
    # 2/3을 쉽게 찍지 않도록: 0/1 기반으로 시작
    pred = np.where(pred == 2, 1, pred)  # 기본값 중상은 경상 쪽으로
    pred = np.where(pred == 3, 1, pred)  # 기본값 사망도 경상 쪽으로

    # 우선순위 적용(확률이 threshold 넘을 때만 2/3 부여)
    pred = np.where(proba[:, 2] >= t_ser, 2, pred)
    pred = np.where(proba[:, 3] >= t_death, 3, pred)
    # 비사상 vs 경상은 argmax(0,1)로 정리
    p01 = np.argmax(proba[:, :2], axis=1)  # 0 or 1
    pred = np.where((pred != 2) & (pred != 3), p01, pred)
    return pred

# ====== TEST 평가 ======
test_proba = booster.predict(dtest, iteration_range=(0, booster.best_iteration + 1))
y_pred = predict_with_priority(test_proba, t_death, t_ser)

print("\n" + "="*70)
print("[TEST] 4클래스 classification_report (Precision 우선 규칙 적용)")
print("="*70)
print(classification_report(
    y_test, y_pred, digits=3,
    target_names=[class_names[i] for i in range(4)]
))

print("\n[TEST] Confusion Matrix (rows=true, cols=pred)")
cm = confusion_matrix(y_test, y_pred, labels=[0,1,2,3])
cm_df = pd.DataFrame(
    cm,
    index=[f"T:{class_names[i]}" for i in range(4)],
    columns=[f"P:{class_names[i]}" for i in range(4)]
)
print(cm_df)


In [ ]:
# %% [1] 라이브러리 로드 및 한글 폰트 설정
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import platform
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# VS Code Interactive Window에서 한글 깨짐 방지
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# %% [2] 데이터 로드 및 세부 항목별(One-Hot) 변환
# 파일 경로를 본인의 환경에 맞게 수정해주세요.
file_path = '/Users/leeseungbo/Desktop/학회 데이터(CSV)/5차 전처리 데이터(1).csv'
df = pd.read_csv(file_path)

target_names = ['사망발생', '중상발생', '경상발생']
X_raw = df.drop(columns=target_names)

# [중요] 세부 항목별(졸음, 과속 등) 분석을 위해 One-Hot Encoding 적용
X_encoded = pd.get_dummies(X_raw)

# %% [3] 방안 A 전용 통합 분석 함수
def run_plan_a_analysis(target_name, spw, mcw):
    print(f"\n{'='*50}\n [방안 A: 전체 데이터 분석] 타겟: {target_name}\n{'='*50}")
    
    y = df[target_name]
    
    # 7:3 데이터 분할 (불균형 비율 유지를 위해 stratify 적용)
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # 승보님이 제시한 최적 하이퍼파라미터 적용
    model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        colsample_bytree=0.4,
        gamma=0.06,
        subsample=0.7,
        reg_alpha=0.2,
        reg_lambda=1.0,
        scale_pos_weight=spw,  # 클래스 불균형 강력 보정
        min_child_weight=mcw,  # 과적합 억제
        random_state=42,
        eval_metric='logloss'
    )
    
    model.fit(X_train, y_train)
    
    # 성능 평가
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
    print(classification_report(y_test, model.predict(X_test)))
    
    # SHAP 시각화 (세부 항목별 나열)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    plt.figure(figsize=(12, 10))
    # max_display=20으로 상위 20개 세부 요인 노출
    shap.summary_plot(shap_values, X_test, max_display=20, show=False)
    plt.title(f"[{target_name}] 사고 발생 기여도 (Plan A)", fontsize=16)
    plt.show()

# %% [4] 타겟별 실행 (제시하신 가중치 반영)
# 사망발생 (비율 2.36%, 가중치 41.33)
run_plan_a_analysis('사망발생', spw=41.33, mcw=5)

# 중상발생 (비율 1.96%, 가중치 49.95)
run_plan_a_analysis('중상발생', spw=49.95, mcw=7)

# 경상발생 (비율 6.47%, 가중치 14.45)
run_plan_a_analysis('경상발생', spw=14.45, mcw=3)

In [ ]:
# %% [1] 라이브러리 로드 및 한글 폰트 설정
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import platform
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# 한글 깨짐 방지 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# %% [2] 데이터 로드 및 세부 항목별(One-Hot) 변환
file_path = '/Users/leeseungbo/Desktop/학회 데이터(CSV)/5차 전처리 데이터(1).csv'
df = pd.read_csv(file_path)

target_names = ['사망발생', '중상발생', '경상발생']
X_raw = df.drop(columns=target_names)

# 세부 항목별 분석을 위해 One-Hot Encoding 적용
X_encoded = pd.get_dummies(X_raw)

# %% [3] 불균형 최적화 및 에러 수정된 분석 함수
def run_plan_a_optimized(target_name):
    print(f"\n{'='*50}\n [최적화 분석] 타겟: {target_name}\n{'='*50}")
    
    y = df[target_name]
    
    # 7:3 데이터 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # 클래스 불균형 가중치 동적 계산
    count_neg = len(y_train[y_train == 0])
    count_pos = len(y_train[y_train == 1])
    dynamic_spw = count_neg / count_pos
    
    # [수정 포인트] early_stopping_rounds를 생성자 내부로 이동
    model = xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        max_delta_step=1,           # 불균형 데이터 수렴 지원
        colsample_bytree=0.6,
        subsample=0.7,
        reg_alpha=0.2,
        scale_pos_weight=dynamic_spw,
        min_child_weight=7 if target_name != '경상발생' else 3,
        random_state=42,
        eval_metric='aucpr',        # 소수 클래스 탐지 성능 중심
        early_stopping_rounds=50    # 여기에 추가 (에러 해결)
    )
    
    # fit 호출 시에는 eval_set만 전달
    model.fit(
        X_train, y_train, 
        eval_set=[(X_test, y_test)], 
        verbose=False
    )
    
    # 성능 평가
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"[{target_name}] 동적 가중치: {dynamic_spw:.2f}")
    print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
    print(classification_report(y_test, model.predict(X_test)))
    
    # SHAP 시각화
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_values, X_test, max_display=20, show=False)
    plt.title(f"[{target_name}] 사고 발생 세부 기여도 (최적화 버전)", fontsize=16)
    plt.show()

# %% [4] 타겟별 실행
run_plan_a_optimized('사망발생')
run_plan_a_optimized('중상발생')
run_plan_a_optimized('경상발생')